- 날짜 형식 정리
- rating을 분석 가능한 순서형 값으로 변환
- 가격/비율 데이터 확인

In [6]:
import pandas as pd
from pathlib import Path

base_dir = Path.cwd().parent
raw_dir = base_dir / "data" / "raw"
processed_dir = base_dir / "data" / "processed"

processed_dir.mkdir(exist_ok=True)

games = pd.read_csv(raw_dir / "games.csv")
users = pd.read_csv(raw_dir / "users.csv")

#### 날짜형식 변환

In [9]:
games["date_release"] = pd.to_datetime(
    games["date_release"],
    errors="coerce"
)

games["release_year"] = games["date_release"].dt.year
games["release_month"] = games["date_release"].dt.month

In [13]:
print(games["date_release"].min())
print(games["date_release"].max())
print(games["date_release"].isna().sum())

1997-06-30 00:00:00
2023-10-24 00:00:00
0


#### Rating 순서형 변수 생성

In [11]:
rating_order = {
    "Overwhelmingly Negative": 1,
    "Very Negative": 2,
    "Negative": 3,
    "Mostly Negative": 4,
    "Mixed": 5,
    "Mostly Positive": 6,
    "Positive": 7,
    "Very Positive": 8,
    "Overwhelmingly Positive": 9
}

games["rating_score"] = games["rating"].map(rating_order)

In [12]:
print(
    games[["rating", "rating_score"]]
    .drop_duplicates()
    .sort_values("rating_score")
)

                       rating  rating_score
3247  Overwhelmingly Negative             1
8425            Very Negative             2
974                  Negative             3
1316          Mostly Negative             4
3                       Mixed             5
6             Mostly Positive             6
1                    Positive             7
0               Very Positive             8
55    Overwhelmingly Positive             9


#### 가격 관련 변수 확인

In [14]:
games["is_free"] = games["price_final"] == 0

games["price_change"] = (
    games["price_original"] - games["price_final"]
)

games["discount_rate"] = games["discount"] / 100

#### 플랫폼 개수

In [15]:
games["platform_count"] = (
    games[["win", "mac", "linux"]]
    .sum(axis=1)
)

#### 파일 저장

In [16]:
games.to_csv(
    processed_dir / "games_cleaned.csv",
    index=False
)

users.to_csv(
    processed_dir / "users_cleaned.csv",
    index=False
)

In [17]:
print(games.shape)
print(users.shape)
print(games.head())

(50872, 20)
(14306064, 3)
   app_id                              title date_release   win    mac  linux  \
0   13500  Prince of Persia: Warrior Within™   2008-11-21  True  False  False   
1   22364            BRINK: Agents of Change   2011-08-03  True  False  False   
2  113020       Monaco: What's Yours Is Mine   2013-04-24  True   True   True   
3  226560                 Escape Dead Island   2014-11-18  True  False  False   
4  249050            Dungeon of the ENDLESS™   2014-10-27  True   True  False   

          rating  positive_ratio  user_reviews  price_final  price_original  \
0  Very Positive              84          2199         9.99            9.99   
1       Positive              85            21         2.99            2.99   
2  Very Positive              92          3722        14.99           14.99   
3          Mixed              61           873        14.99           14.99   
4  Very Positive              88          8784        11.99           11.99   

   discount 